# RLFinetuneLab: End-to-End Alignment Pipeline (SFT -> DPO -> GRPO)

**Author:** Amaan Gupta  
**Repository:** [github.com/amaannn08/RLFinetuneLab](https://github.com/amaannn08/RLFinetuneLab)  
**Target Hardware:** Free Google Colab T4 GPU (15GB VRAM) or A100  
**Target Model:** `Qwen/Qwen2.5-0.5B-Instruct` or `Qwen/Qwen2.5-1.5B-Instruct`

---
### Pipeline Overview
This notebook demonstrates the complete, end-to-end post-training alignment pipeline:
1. **Supervised Fine-Tuning (SFT)** with 4-bit QLoRA and FlashAttention/SDPA
2. **Direct Preference Optimization (DPO)** with implicit reward modeling and adapter-frozen reference policy
3. **Group Relative Policy Optimization (GRPO)** with DeepSeek-R1 style verifiable math accuracy & XML `<think>` format rewards
4. **Benchmarking & Pairwise Evaluation** (Perplexity + Win-rate judge)
5. **LoRA Weight Fusion & GGUF Export** for high-throughput vLLM / llama.cpp serving

## 1. Environment Setup & Hardware Verification

In [ ]:
# Check CUDA GPU
!nvidia-smi

# Install RLFinetuneLab & dependencies
!pip install -q --upgrade pip
!pip install -q torch transformers peft trl accelerate datasets bitsandbytes pydantic pyyaml rich

# Clone repo
!git clone https://github.com/amaannn08/RLFinetuneLab.git
%cd RLFinetuneLab
!pip install -q -e .

In [ ]:
from rlfinetunelab.utils.hardware import get_hardware_summary
import pprint

hw = get_hardware_summary()
pprint.pprint(hw)
assert hw["cuda_available"], "Please enable GPU acceleration in Colab: Runtime -> Change runtime type -> T4 GPU"

## 2. Stage 1: Supervised Fine-Tuning (SFT) with 4-bit QLoRA
We start by fine-tuning the base conversational model on multi-turn instruction datasets.

In [ ]:
from rlfinetunelab.config import load_config
from rlfinetunelab.trainers.sft_trainer import run_sft_training

# Load pre-configured SFT QLoRA spec
sft_cfg = load_config(
    "configs/sft/qwen2.5_0.5b_qlora.yaml",
    overrides=[
        "sft.num_train_epochs=1",
        "sft.max_steps=50",
        "dataset.max_train_samples=200",
        "dataset.max_eval_samples=50",
        "sft.output_dir=outputs/colab_sft"
    ]
)

sft_results = run_sft_training(sft_cfg)
print("SFT Complete! Final Loss:", sft_results["training_loss"])

## 3. Stage 2: Direct Preference Optimization (DPO)
Aligning the model with human preferences using implicit log-ratio rewards without a separate reward model.

In [ ]:
from rlfinetunelab.trainers.dpo_trainer import run_dpo_training

dpo_cfg = load_config(
    "configs/dpo/qwen2.5_0.5b_dpo.yaml",
    overrides=[
        "dpo.num_train_epochs=1",
        "dpo.max_steps=30",
        "dataset.max_train_samples=150",
        "dataset.max_eval_samples=30",
        "dpo.output_dir=outputs/colab_dpo"
    ]
)

dpo_results = run_dpo_training(dpo_cfg)
print("DPO Alignment Complete! Loss:", dpo_results["training_loss"])

## 4. Stage 3: Group Relative Policy Optimization (GRPO)
Applying DeepSeek-R1 / DeepSeekMath style reasoning RL with verifiable accuracy & `<think>` formatting rewards.

In [ ]:
from rlfinetunelab.trainers.grpo_trainer import run_grpo_training

grpo_cfg = load_config(
    "configs/grpo/qwen2.5_0.5b_grpo.yaml",
    overrides=[
        "grpo.num_train_epochs=1",
        "grpo.max_steps=20",
        "grpo.num_generations=2",
        "dataset.max_train_samples=50",
        "grpo.output_dir=outputs/colab_grpo"
    ]
)

grpo_results = run_grpo_training(grpo_cfg)
print("GRPO Training Complete! Loss:", grpo_results["training_loss"])

## 5. Benchmarking & Pairwise Win-Rate Evaluation
We evaluate the aligned model against the base instruction baseline.

In [ ]:
from rlfinetunelab.evaluation.pairwise_judge import compute_pairwise_win_rate

prompts = [
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did she sell altogether in April and May?",
    "Weng earns $12 an hour for babysitting. Yesterday, she babysat for 5 hours. How much did she earn?",
]
targets = ["72", "60"]

# Sample candidate outputs
outputs_rl = [
    "<think>In April Natalia sold 48 clips. In May she sold 48/2 = 24 clips. Altogether: 48 + 24 = 72.</think>\n<answer>72</answer>",
    "<think>Weng earns 12 dollars per hour. For 5 hours: 12 * 5 = 60 dollars.</think>\n<answer>60</answer>"
]
outputs_base = [
    "She sold 48 clips in April and 24 in May. Total is 72 clips.",
    "Weng earned 60 dollars."
]

pairwise_res = compute_pairwise_win_rate(
    prompts=prompts,
    completions_a=outputs_rl,
    completions_b=outputs_base,
    targets=targets,
    name_a="RL_Reasoner",
    name_b="Base_Instruct"
)
print("Win Rate RL Reasoner:", pairwise_res["RL_Reasoner_win_rate"], "%")

## 6. Adapter Merging & GGUF Conversion for Serving
Merge the trained PEFT adapter into 16-bit base weights and convert to GGUF for llama.cpp / Ollama local deployment.

In [ ]:
from rlfinetunelab.export.merge_lora import merge_lora_to_base

# Merge LoRA adapter into 16-bit consolidated model
merged_path = merge_lora_to_base(
    base_model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    adapter_path="outputs/colab_sft/final_adapter",
    output_dir="outputs/merged_qwen_0.5b",
    torch_dtype="float16"
)
print("Merged model ready at:", merged_path)

# Export commands for GGUF
from rlfinetunelab.export.convert_gguf import generate_gguf_conversion_command
cmds = generate_gguf_conversion_command(
    model_dir="outputs/merged_qwen_0.5b",
    output_gguf_path="outputs/qwen2.5_0.5b_q4_k_m.gguf",
    quantization_type="Q4_K_M"
)
print("\nGGUF Export Commands:")
for c in cmds:
    print(c)